# Visualizing Path Graphs for BFS Evaluation

This notebook generates and visualizes **path graphs** used in the `tests/eval_path.py` script.

Path graphs are worst-case topologies for Breadth-First Search (BFS). A path of $n$ nodes has a maximum BFS depth of $n-1$ when the source is at an endpoint. By testing the model on path graphs of varying lengths, we can evaluate its robustness and autoregressive extrapolation capabilities at specific algorithmic depths.

In [ ]:
import sys
import os

# Add project root to path so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch

# PyTorch 2.6 changed the default of weights_only to True, which breaks PyG datasets
# We monkey-patch it here to avoid a whack-a-mole with safe_globals
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_load(*args, **kwargs)
torch.load = _patched_load

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# Import SALSACLRS dataset class
from salsaclrs import SALSACLRSDataset

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
def parse_bfs_sample(data):
    """Extract graph structure and BFS results from a CLRSData sample.
    
    Returns:
        G: undirected NetworkX graph
        source: source node index
        bfs_tree_edges: set of (u, v) tuples — directed predecessor edges (child -> parent)
        num_nodes: number of nodes
    """
    edge_index = data.edge_index.numpy()
    num_nodes = data.s.shape[0]
    
    # Build undirected graph
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    edges = set()
    for i in range(edge_index.shape[1]):
        u, v = edge_index[0, i], edge_index[1, i]
        edges.add((min(u, v), max(u, v)))
    G.add_edges_from(edges)
    
    # Source node
    source = data.s.argmax().item()
    
    # BFS tree edges from pi: edges where pi == 1.0
    # pi is [E] — one-hot mask over edge_index
    # edge_index[:, i] = (src, dst) where pi[i] == 1 means dst's predecessor is src
    pi = data.pi.numpy()
    bfs_tree_edges = []
    for i in range(len(pi)):
        if pi[i] > 0.5:
            u, v = edge_index[0, i], edge_index[1, i]
            # Skip self-loops (source points to itself)
            if u != v:
                bfs_tree_edges.append((u, v))
    
    return G, source, bfs_tree_edges, num_nodes

def draw_bfs_graph(data, ax=None, title=None, layout="kamada_kawai", seed=42):
    """Visualize a SALSA-CLRS BFS sample showing the graph and BFS tree.
    
    Args:
        data: CLRSData sample
        ax: matplotlib axes (created if None)
        title: optional title string
        layout: 'kamada_kawai', 'spring', or 'spectral'
        seed: random seed for spring layout
    """
    G, source, bfs_tree_edges, num_nodes = parse_bfs_sample(data)
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Layout - For path graphs, Kamada-Kawai or spectral usually works best
    if layout == "kamada_kawai":
        pos = nx.kamada_kawai_layout(G)
    elif layout == "spring":
        pos = nx.spring_layout(G, seed=seed, k=2.0/np.sqrt(num_nodes))
    else:
        pos = nx.spectral_layout(G)
    
    # Draw all edges (light gray)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#d0d0d0", width=1.0, alpha=0.8)
    
    # Draw BFS tree edges as directed arrows (child → parent via predecessor)
    if bfs_tree_edges:
        tree_G = nx.DiGraph()
        tree_G.add_nodes_from(G.nodes())
        tree_G.add_edges_from(bfs_tree_edges)
        nx.draw_networkx_edges(
            tree_G, pos, edgelist=bfs_tree_edges, ax=ax,
            edge_color="#e63946", width=2.5, alpha=0.9,
            arrows=True, arrowstyle='-|>', arrowsize=15,
            connectionstyle='arc3,rad=0.1', min_source_margin=10, min_target_margin=10
        )
    
    # Determine which nodes are in the BFS tree
    tree_nodes = set()
    tree_nodes.add(source)
    for u, v in bfs_tree_edges:
        tree_nodes.add(u)
        tree_nodes.add(v)
    
    unreachable = [n for n in G.nodes() if n not in tree_nodes]
    reachable = [n for n in G.nodes() if n in tree_nodes and n != source]
    
    # Draw unreachable nodes
    if unreachable:
        nx.draw_networkx_nodes(G, pos, nodelist=unreachable, ax=ax,
                               node_color="#bbb", node_size=300, edgecolors="#888", linewidths=1.0)
    
    # Draw reachable nodes
    if reachable:
        nx.draw_networkx_nodes(G, pos, nodelist=reachable, ax=ax,
                               node_color="#457b9d", node_size=300, edgecolors="#1d3557", linewidths=1.5)
    
    # Draw source node (larger, distinct color)
    nx.draw_networkx_nodes(G, pos, nodelist=[source], ax=ax,
                           node_color="#f4a261", node_size=500, edgecolors="#e76f51", 
                           linewidths=2.5, node_shape="*")
    
    # Node labels
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="white", font_weight="bold")
    
    # Legend
    legend_elements = [
        Line2D([0], [0], marker='*', color='w', markerfacecolor='#f4a261', 
               markeredgecolor='#e76f51', markersize=15, label='Source'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#457b9d',
               markeredgecolor='#1d3557', markersize=10, label='Reachable'),
        Line2D([0], [0], color='#e63946', linewidth=2.5, label='BFS tree edge (→ parent)'),
        Line2D([0], [0], color='#d0d0d0', linewidth=1.0, label='Graph edge'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=7, framealpha=0.9)
    
    if title is None:
        title = f"Path Graph (N={num_nodes}), Max Depth={data.length}"
    ax.set_title(title, fontsize=12)
    ax.axis('off')
    
    return pos

## Generate and Visualize Path Graphs

We will generate path graphs of size 16, 32, and 64 nodes. These correspond to depths of 15, 31, and 63 respectively.

In [ ]:
# Set a fixed seed for reproducible data generation
SEED = 42
data_root = os.path.join(project_root, "data", "salsaclrs", f"seed_{SEED}")

node_sizes = [16, 32, 64]
fig, axes = plt.subplots(3, 1, figsize=(12, 18))

for i, n_nodes in enumerate(node_sizes):
    print(f"Generating Path Graph dataset for N={n_nodes}...")
    
    dataset = SALSACLRSDataset(
        root=data_root,
        split="test",
        algorithm="bfs",
        num_samples=1,  # We just need 1 sample for visualization
        graph_generator="path",
        graph_generator_kwargs={"n": n_nodes},
        verify_duplicates=False,
        ignore_all_hints=True,  # We just want the inputs and outputs
        nickname=f"path_d{n_nodes-1}",
    )
    
    sample = dataset[0]
    ax = axes[i]
    
    draw_bfs_graph(
        sample, 
        ax=ax, 
        title=f"Path Graph: {n_nodes} nodes (Max Depth: {sample.length})",
        layout="kamada_kawai"
    )

plt.tight_layout()
plt.show()